# IC Quant Transformer - v3

Based on **Transformer v2** with two additions:

1. **Cross-asset features** - each target dataset now includes the other targets' `ret_1d` as predictors.
2. **Volatility-normalized target** - model predicts `log(close_{t+HORIZON} / close_t) / rolling_std(ret_1d, VOL_WINDOW)`. This amplifies signal-to-noise on daily returns. `predict_next()` multiplies by current volatility to recover an actual return estimate.

`HORIZON=5`, `VOL_WINDOW=20`. Transformer architecture, HuberLoss, and training loop are unchanged.


# Google Colab Setting

In [ ]:
!pip install -q "numpy<2.0" "pandas<2.2" refinitiv-data fredapi optuna xgboost tqdm scikit-learn scipy

# Package Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

In [ ]:
import pandas as pd
import numpy as np
import refinitiv.data as rd
from fredapi import Fred

import datetime as dt
import json
import os
import scipy.stats as stats

from xgboost import XGBRegressor
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import optuna
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Data Acquisition

In [ ]:
START = "2000-01-01"
END = dt.datetime.today().strftime("%Y-%m-%d")

REFINITIV_TICKERS = ["SPY", "TLT.O", "GLD", "XLE", ".DXY"]
TARGET_TICKERS    = ["SPY", "TLT.O", "GLD", "XLE"]

FRED_TICKERS = {
    "T10Y2Y":     "Term_Spread",
    "BAA10Y":     "Credit_Spread",
    "T10YIE":     "Breakeven_Inflation",
    "VIXCLS":     "VIX",
    "NASDAQXAU":  "Gold_Spot",
    "DCOILWTICO": "WTI_Crude",
}

OUT_PATH    = "../Data/Index/merged_daily.csv"
DATASET_DIR = "../data"

MACRO_LEVEL_DIFF = ["Term_Spread", "Credit_Spread", "Breakeven_Inflation", "VIX"]
MACRO_LOG_RETURN = ["Gold_Spot", "WTI_Crude"]

In [ ]:
def key_loader(filepath):
    try:
        with open(filepath) as f:
            app_key = json.load(f)['sessions']['platform']['rdp']['app-key']
        print("Key loaded successfully")
        return app_key
    except FileNotFoundError:
        print(f"Error: '{filepath}' not found.")
        raise
    except KeyError as e:
        print(f"Error: missing key {e}.")
        raise

In [ ]:
def check_state(state, message, session):
    print(f"State: {state}")
    print(f"Message: {message}")
    print("\n")

In [ ]:
def fetch_refinitiv_data(tickers, start_date, end_date, app_key):
    session = None
    try:
        session = rd.session.desktop.Definition(app_key=app_key).get_session()
        rd.session.set_default(session)
        session.on_state(check_state)
        session.open()

        fields = ["TR.PriceOpen", "TR.PriceHigh", "TR.PriceLow", "TR.PriceClose", "TR.Volume"]
        frames = []
        for ticker in tickers:
            print(f"[REFINITIV] Fetching {ticker} ...")
            try:
                tdf = rd.get_history(
                    universe=[ticker],
                    fields=fields,
                    interval="1D",
                    start=start_date,
                    end=end_date,
                )
                if tdf is None or tdf.empty:
                    print(f"[REFINITIV] {ticker}: EMPTY"); continue
                tdf.index = pd.to_datetime(tdf.index)
                try:
                    tdf.index = tdf.index.tz_localize(None)
                except (TypeError, AttributeError):
                    pass
                tdf = tdf[~tdf.index.duplicated(keep="last")]
                if isinstance(tdf.columns, pd.MultiIndex):
                    tdf.columns = [f"{tic}_{fld}" for tic, fld in tdf.columns]
                else:
                    tdf.columns = [f"{ticker}_{col}" for col in tdf.columns]
                print(f"[REFINITIV] {ticker}: OK | rows={len(tdf)}")
                frames.append(tdf)
            except Exception as e:
                print(f"[REFINITIV] {ticker}: FAIL | {e}")

        if not frames:
            raise ValueError("No Refinitiv data was fetched successfully.")
        return pd.concat(frames, axis=1).sort_index()
    finally:
        if session is not None:
            session.close()

In [ ]:
def fetch_fred_data(tickers_dict, start_date, end_date):
    api_key = os.environ.get("FRED_API_KEY")
    if not api_key:
        with open("secrets.json") as f:
            api_key = json.load(f)["fred_api_key"]
    fred = Fred(api_key=api_key)
    combined = pd.DataFrame()
    for ticker, name in tickers_dict.items():
        print(f"[FRED] Fetching {ticker} ({name}) ...")
        try:
            series = fred.get_series(ticker, observation_start=start_date, observation_end=end_date)
            if series is None or len(series) == 0:
                print(f"[FRED] {ticker}: EMPTY"); continue
            combined[name] = series
            print(f"[FRED] {ticker}: OK | rows={len(series)}")
        except Exception as e:
            print(f"[FRED] {ticker}: FAIL | {e}")
    combined.index = pd.to_datetime(combined.index)
    return combined

In [ ]:
def get_raw_dataset(start_date=START, end_date=END, filepath="refinitiv-data.config.json"):
    app_key  = key_loader(filepath)
    df_market = fetch_refinitiv_data(REFINITIV_TICKERS, start_date, end_date, app_key)
    df_macro  = fetch_fred_data(FRED_TICKERS, start_date, end_date)
    df_merged = df_market.join(df_macro, how="left")
    df_merged = df_merged.apply(pd.to_numeric, errors="coerce").ffill()
    close_cols = [f"{t}_Price Close" for t in TARGET_TICKERS if f"{t}_Price Close" in df_merged.columns]
    df_merged  = df_merged.dropna(subset=close_cols, how="any")
    df_merged  = df_merged.drop([".DXY_Volume"], axis=1, errors="ignore")
    return df_merged

In [ ]:
filepath = "refinitiv-data.config.json"
df = get_raw_dataset(filepath=filepath)
print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

In [ ]:
df

In [ ]:
df.to_csv(OUT_PATH)

# Data Preprocessing

In [ ]:
df = pd.read_csv(OUT_PATH, index_col=0, parse_dates=True)

In [ ]:
def transform_macro(df, level_diff_cols, log_return_cols):
    out = pd.DataFrame(index=df.index)
    for col in level_diff_cols:
        if col in df.columns:
            out[f"{col}_level"] = df[col]
            out[f"{col}_diff"]  = df[col].diff()
    for col in log_return_cols:
        if col in df.columns:
            out[f"{col}_logret"] = np.log(
                df[col].clip(lower=1e-10) / df[col].shift(1).clip(lower=1e-10))
    if ".DXY_Price Close" in df.columns:
        dxy = pd.to_numeric(df[".DXY_Price Close"], errors="coerce")
        out["DXY_logret"] = np.log(dxy.clip(lower=1e-10) / dxy.shift(1).clip(lower=1e-10))
    return out

# Feature Engineering

In [ ]:
def compute_technical_features(close):
    out = pd.DataFrame(index=close.index)
    ret_1d = np.log(close / close.shift(1))
    out["ret_1d"]  = ret_1d
    out["ret_5d"]  = np.log(close / close.shift(5))
    out["ret_20d"] = np.log(close / close.shift(20))
    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    out["macd"]    = ema_12 - ema_26
    delta = close.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    out["rsi_14"]  = 100 - 100 / (1 + gain / loss)
    out["vol_20d"] = ret_1d.rolling(20).std() * np.sqrt(252)
    return out

def build_cross_asset_returns(df, targets):
    """v3: cross-asset daily log returns for feature augmentation."""
    out = {}
    for t in targets:
        close = df[f"{t}_Price Close"]
        out[f"{t}_ret_1d"] = np.log(close / close.shift(1))
    return pd.DataFrame(out, index=df.index)

def build_target_dataset(df, target, macro_features, cross_rets):
    """v3: cross-asset features + volatility-normalized t+HORIZON log return target.

    target_t = log(close_{t+H} / close_t) / rolling_std(ret_1d, VOL_WINDOW)

    Model output is a realised-Sharpe-like quantity; `predict_next()` multiplies
    by current volatility to recover an actual return estimate.
    """
    close = df[f"{target}_Price Close"]
    tech  = compute_technical_features(close).add_prefix(f"{target}_")
    cross = cross_rets.drop(columns=[f"{target}_ret_1d"], errors="ignore")

    raw_ret  = np.log(close.shift(-HORIZON) / close)
    ret_1d   = np.log(close / close.shift(1))
    roll_vol = ret_1d.shift(1).rolling(VOL_WINDOW).std().clip(lower=1e-6)
    y        = (raw_ret / roll_vol).rename("target")

    return pd.concat([tech, macro_features, cross, y], axis=1).dropna()


In [ ]:
macro_features = transform_macro(df, MACRO_LEVEL_DIFF, MACRO_LOG_RETURN)
cross_rets     = build_cross_asset_returns(df, TARGET_TICKERS)   # v3: cross-asset returns
datasets = {t: build_target_dataset(df, t, macro_features, cross_rets) for t in TARGET_TICKERS}

os.makedirs(DATASET_DIR, exist_ok=True)
for target, data in datasets.items():
    safe = target.replace(".", "_")
    data.to_csv(f"{DATASET_DIR}/{safe}_dataset.csv")
    print(f"Saved: {safe}_dataset.csv | shape={data.shape}")

for t, data in datasets.items():
    print(f"{t}: shape={data.shape}, range={data.index.min().date()} -> {data.index.max().date()}")


# Feature Selection

In [ ]:
# Load datasets from CSV
datasets = {}
for target in TARGET_TICKERS:
    safe = target.replace(".", "_")
    path = f"{DATASET_DIR}/{safe}_dataset.csv"
    datasets[target] = pd.read_csv(path, index_col=0, parse_dates=True)
    print(f"Loaded: {safe}_dataset.csv | shape={datasets[target].shape}")

## Correlation Test

In [ ]:
SPEARMAN_THRESHOLD = 0.90
TOP_K = 12

def stage1_redundancy_filter(X, y, threshold=SPEARMAN_THRESHOLD):
    corr     = X.corr(method="spearman").abs()
    dist     = (1 - corr).clip(lower=0)
    dist_arr = dist.values.copy()
    np.fill_diagonal(dist_arr, 0)
    condensed = squareform(dist_arr, checks=False)
    Z      = linkage(condensed, method="ward")
    labels = fcluster(Z, t=1 - threshold, criterion="distance")
    ic     = X.corrwith(y, method="spearman").abs()
    selected = []
    for cid in np.unique(labels):
        cols = X.columns[labels == cid].tolist()
        selected.append(ic[cols].idxmax())
    return selected

## XGBoost Feature Importance

In [ ]:
def stage2_relevance_filter(X, y, top_k=TOP_K):
    xgb_device = "cuda" if torch.cuda.is_available() else "cpu"
    model = XGBRegressor(
        max_depth=3, n_estimators=100, random_state=42, verbosity=0,
        device=xgb_device, tree_method="hist",
    )
    model.fit(X, y)
    importances = pd.Series(model.feature_importances_, index=X.columns)
    return importances.nlargest(top_k)  # Series (feature → importance)

def select_features(X, y, top_k=TOP_K):
    stage1 = stage1_redundancy_filter(X, y)
    stage2 = stage2_relevance_filter(X[stage1], y, top_k=top_k)
    return stage2.index.tolist(), stage2  # (feature_list, importance_series)

# Transformer Model

In [ ]:
LOOKBACK            = 20
INITIAL_TRAIN_YEARS = 5
TEST_SIZE           = 60
EMBARGO             = LOOKBACK
HORIZON             = 5    # v3: t+5 prediction horizon
VOL_WINDOW          = 20   # v3: rolling vol window for target normalization

def make_sequence_dataset(data, feature_cols, lookback=LOOKBACK):
    X = data[feature_cols].values.astype(np.float32)
    y = data["target"].values.astype(np.float32)
    xs, ys = [], []
    for i in range(lookback, len(data)):
        xs.append(X[i - lookback:i])
        ys.append(y[i])
    if not xs:
        return torch.empty(0, lookback, len(feature_cols)), torch.empty(0)
    return torch.tensor(np.array(xs)), torch.tensor(np.array(ys))

def expanding_window_folds(n, initial_train, test_size, embargo, purge=0):
    folds = []
    train_end = initial_train
    while train_end + embargo + test_size <= n:
        train_idx  = np.arange(0, max(train_end - purge, 1))
        test_start = train_end + embargo
        test_end   = test_start + test_size
        test_idx   = np.arange(test_start, test_end)
        folds.append((train_idx, test_idx))
        train_end += test_size
    return folds

def expanding_window_scale(X_arr, min_periods=50):
    """Normalize each training sample using only past data (causal)."""
    n, d    = X_arr.shape
    X_arr   = X_arr.astype(np.float64)
    out     = np.empty((n, d), dtype=np.float32)
    mp      = min(min_periods, n)
    warmup_mean = X_arr[:mp].mean(axis=0)
    warmup_std  = X_arr[:mp].std(axis=0).clip(min=1e-8)
    cum_sum = np.zeros(d)
    cum_sq  = np.zeros(d)
    for i in range(n):
        if i < mp:
            out[i] = ((X_arr[i] - warmup_mean) / warmup_std).astype(np.float32)
        else:
            mean = cum_sum / i
            var  = (cum_sq / i - mean ** 2).clip(min=1e-8)
            out[i] = ((X_arr[i] - mean) / np.sqrt(var)).astype(np.float32)
        cum_sum += X_arr[i]
        cum_sq  += X_arr[i] ** 2
    full_mean = X_arr.mean(axis=0)
    full_std  = X_arr.std(axis=0).clip(min=1e-8)
    return out, full_mean, full_std

In [ ]:
class TransformerBlock(nn.Module):
    """Transformer Encoder Block with Pre-Layer Normalization and GELU FFN."""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.attn  = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout_rate, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim),
        )
        self.drop1 = nn.Dropout(dropout_rate)
        self.drop2 = nn.Dropout(dropout_rate)

    def forward(self, x):
        # Pre-LN attention
        xn = self.norm1(x)
        a, _ = self.attn(xn, xn, xn)
        x = x + self.drop1(a)
        # Pre-LN FFN
        xn = self.norm2(x)
        return x + self.drop2(self.ffn(xn))


class LearnablePositionalEmbedding(nn.Module):
    """Learnable positional embedding (modern approach)."""
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, max_len, embed_dim))
        nn.init.trunc_normal_(self.pe, std=0.02)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding (original Transformer approach)."""
    def __init__(self, max_len, embed_dim):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, embed_dim, 2).float() * -(np.log(10000.0) / embed_dim))
        pe = torch.zeros(1, max_len, embed_dim)
        pe[0, :, 0::2] = torch.sin(pos * div)
        pe[0, :, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerRegressor(nn.Module):
    """
    Transformer encoder for t+1 log return prediction.
    Input projection → Positional Encoding → N×TransformerBlock →
    LayerNorm → GlobalAvgPool → MLP head
    """
    def __init__(self, num_features, embed_dim=64, num_heads=4, ff_dim=256,
                 num_blocks=2, dropout_rate=0.1, use_learnable_pe=True,
                 lookback=LOOKBACK):
        super().__init__()
        self.input_proj = nn.Linear(num_features, embed_dim)
        self.pos_enc = (
            LearnablePositionalEmbedding(lookback, embed_dim)
            if use_learnable_pe
            else SinusoidalPositionalEncoding(lookback, embed_dim)
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.blocks  = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout_rate)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 64), nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 32),        nn.GELU(), nn.Dropout(dropout_rate),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.input_proj(x)   # (B, T, embed_dim)
        x = self.pos_enc(x)
        x = self.dropout(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        x = x.mean(dim=1)        # Global average pool over time
        return self.head(x).squeeze(-1)

# Training Setup

In [ ]:
if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
    torch.set_num_threads(4)

print(f"Device: {DEVICE}")

DEFAULT_PARAMS = {
    "embed_dim":        64,
    "num_heads":        4,
    "ff_dim":           256,
    "num_blocks":       2,
    "dropout_rate":     0.1,
    "lr":               1e-3,
    "batch_size":       64,
    "use_learnable_pe": True,
}

EPOCHS        = 50
PATIENCE      = 7
OPTUNA_TRIALS = 30
OPTUNA_EPOCHS = 15
SAVE_DIR      = "../model"

In [ ]:
def prepare_fold(data, train_idx, test_idx, feat_cols, lookback=LOOKBACK):
    """
    v2: feat_cols fixed globally (no per-fold selection).
    Scaler: expanding window StandardScaler on train, full-train stats on test.
    """
    train_df = data.iloc[train_idx]
    X_tr_raw = train_df[feat_cols].values.astype(np.float64)
    X_tr_scaled, full_mean, full_std = expanding_window_scale(X_tr_raw)

    train_scaled = train_df.copy()
    train_scaled[feat_cols] = X_tr_scaled

    test_start = test_idx[0]
    test_end   = test_idx[-1] + 1
    test_slice = data.iloc[test_start - lookback:test_end].copy()
    test_slice[feat_cols] = (
        (test_slice[feat_cols].values - full_mean) / full_std
    ).astype(np.float32)

    X_tr, y_tr = make_sequence_dataset(train_scaled, feat_cols, lookback)
    X_te, y_te = make_sequence_dataset(test_slice,   feat_cols, lookback)

    X_tr = X_tr.to(DEVICE); y_tr = y_tr.to(DEVICE)
    X_te = X_te.to(DEVICE); y_te = y_te.to(DEVICE)
    return X_tr, y_tr, X_te, y_te, full_mean, full_std

In [ ]:
def train_one_fold(X_tr, y_tr, X_te, y_te, params=None,
                   epochs=EPOCHS, patience=PATIENCE):
    p          = {**DEFAULT_PARAMS, **(params or {})}
    batch_size = int(p["batch_size"])

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size, shuffle=False)

    model = TransformerRegressor(
        num_features    = X_tr.shape[-1],
        embed_dim       = int(p["embed_dim"]),
        num_heads       = int(p["num_heads"]),
        ff_dim          = int(p["ff_dim"]),
        num_blocks      = int(p["num_blocks"]),
        dropout_rate    = float(p["dropout_rate"]),
        use_learnable_pe= bool(p["use_learnable_pe"]),
        lookback        = X_tr.shape[1],
    ).to(DEVICE)
    if DEVICE == "cuda":
        model = torch.compile(model)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=float(p["lr"]), weight_decay=1e-5)
    criterion = nn.HuberLoss(delta=0.01)
    use_amp   = (DEVICE == "cuda")
    amp_ctx   = torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_amp)

    best_val_loss  = float("inf")
    best_state     = None
    patience_count = 0
    train_losses, val_losses = [], []

    for _ in range(epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            with amp_ctx:
                pred = model(xb)
                loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item() * len(xb)
        train_losses.append(ep_loss / len(X_tr))

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                with amp_ctx:
                    pred = model(xb)
                val_loss += criterion(pred.float(), yb.float()).item() * len(xb)
        val_loss /= len(X_te)
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            best_state = {
                k: v.cpu().clone() for k, v in
                (model._orig_mod if hasattr(model, "_orig_mod") else model).state_dict().items()
            }
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    (model._orig_mod if hasattr(model, "_orig_mod") else model).load_state_dict(best_state)

    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in test_loader:
            with amp_ctx:
                p_out = model(xb)
            preds.append(p_out.float().cpu())
    preds = torch.cat(preds).numpy() if preds else np.array([])
    return model, preds, train_losses, val_losses

In [ ]:
def tune_with_optuna(data, folds, feat_cols, n_trials=OPTUNA_TRIALS):
    train_idx, test_idx = folds[0]
    X_tr, y_tr, X_te, y_te, _, _ = prepare_fold(data, train_idx, test_idx, feat_cols)
    y_te_np = y_te.cpu().numpy()

    def objective(trial):
        embed_dim = trial.suggest_categorical("embed_dim", [64, 128, 256])
        num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
        # embed_dim must be divisible by num_heads
        if embed_dim % num_heads != 0:
            raise optuna.exceptions.TrialPruned()
        params = {
            "embed_dim":        embed_dim,
            "num_heads":        num_heads,
            "ff_dim":           trial.suggest_categorical("ff_dim", [256, 512]),
            "num_blocks":       trial.suggest_int("num_blocks", 2, 4),
            "dropout_rate":     trial.suggest_float("dropout_rate", 0.1, 0.4, step=0.05),
            "lr":               trial.suggest_float("lr", 1e-4, 1e-3, log=True),
            "use_learnable_pe": trial.suggest_categorical("use_learnable_pe", [True, False]),
            "batch_size":       trial.suggest_categorical("batch_size", [32, 64, 128]),
        }
        _, preds, _, _ = train_one_fold(
            X_tr, y_tr, X_te, y_te, params=params, epochs=OPTUNA_EPOCHS)
        if len(preds) == 0:
            return float("inf")
        return float(((preds - y_te_np) ** 2).mean())

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.tight_layout(); plt.show()
    optuna.visualization.matplotlib.plot_param_importances(study)
    plt.tight_layout(); plt.show()

    return study

In [ ]:
def walk_forward(data, params, folds, feat_cols, desc="folds"):
    """v2: feat_cols fixed globally. No per-fold feature selection."""
    all_preds, all_truth, all_dates = [], [], []
    fold_history = []
    last_model, last_mean, last_std = None, None, None
    last_train_losses, last_val_losses = [], []

    for k, (train_idx, test_idx) in enumerate(tqdm(folds, desc=desc, leave=False)):
        X_tr, y_tr, X_te, y_te, full_mean, full_std = prepare_fold(
            data, train_idx, test_idx, feat_cols)
        if len(X_tr) == 0 or len(X_te) == 0:
            continue
        model, preds, train_losses, val_losses = train_one_fold(
            X_tr, y_tr, X_te, y_te, params=params, epochs=EPOCHS)
        truth = y_te.cpu().numpy()
        all_preds.append(preds)
        all_truth.append(truth)
        all_dates.append(data.index[test_idx])
        last_model, last_mean, last_std = model, full_mean, full_std
        last_train_losses, last_val_losses = train_losses, val_losses

        fold_mse = float(((preds - truth) ** 2).mean()) if len(preds) else float("nan")
        fold_hit = float(((preds * truth) > 0).mean())  if len(preds) else float("nan")
        fold_history.append({
            "fold":       k,
            "train_end":  data.index[train_idx[-1]],
            "test_start": data.index[test_idx[0]],
            "test_end":   data.index[test_idx[-1]],
            "n_train": len(X_tr), "n_test": len(X_te),
            "mse": fold_mse, "hit": fold_hit,
        })

    preds = np.concatenate(all_preds) if all_preds else np.array([])
    truth = np.concatenate(all_truth) if all_truth else np.array([])
    if all_dates:
        dates = pd.DatetimeIndex(np.concatenate([d.values for d in all_dates]))[:len(preds)]
    else:
        dates = pd.DatetimeIndex([])

    preds_df       = pd.DataFrame({"pred": preds, "truth": truth}, index=dates)
    fold_history_df = pd.DataFrame(fold_history)
    return (preds_df, last_model, last_mean, last_std,
            fold_history_df, last_train_losses, last_val_losses)

In [ ]:
# SAVE_DIR = "/content/drive/MyDrive/transformer_results"  # Google Colab
SAVE_DIR = "../model"

def save_target(target, result, params, save_dir=SAVE_DIR):
    safe = target.replace(".", "_")
    result["preds"].to_csv(f"{save_dir}/{safe}_preds.csv")
    result["fold_history"].to_csv(f"{save_dir}/{safe}_fold_history.csv", index=False)
    _model = result["last_model"]
    _sd    = (_model._orig_mod.state_dict()
              if hasattr(_model, "_orig_mod") else _model.state_dict())
    torch.save(_sd, f"{save_dir}/{safe}_model.pt")
    with open(f"{save_dir}/{safe}_scaler.json", "w") as f:
        json.dump({"mean": result["last_mean"].tolist(),
                   "std":  result["last_std"].tolist()}, f)
    with open(f"{save_dir}/{safe}_feats.json", "w") as f:
        json.dump(result["last_feats"], f)
    with open(f"{save_dir}/{safe}_params.json", "w") as f:
        json.dump(params, f, indent=2)
    print(f"  [{target}] saved to {save_dir}/")


def load_results(save_dir=SAVE_DIR):
    loaded = {}
    for target in TARGET_TICKERS:
        safe = target.replace(".", "_")
        preds_df = pd.read_csv(
            f"{save_dir}/{safe}_preds.csv", index_col=0, parse_dates=True)
        with open(f"{save_dir}/{safe}_feats.json") as f:
            feats = json.load(f)
        with open(f"{save_dir}/{safe}_scaler.json") as f:
            sc = json.load(f)
        feat_mean = np.array(sc["mean"])
        feat_std  = np.array(sc["std"])
        with open(f"{save_dir}/{safe}_params.json") as f:
            best = {**DEFAULT_PARAMS, **json.load(f)}

        model = TransformerRegressor(
            num_features    = len(feats),
            embed_dim       = int(best["embed_dim"]),
            num_heads       = int(best["num_heads"]),
            ff_dim          = int(best["ff_dim"]),
            num_blocks      = int(best["num_blocks"]),
            dropout_rate    = float(best["dropout_rate"]),
            use_learnable_pe= bool(best["use_learnable_pe"]),
            lookback        = LOOKBACK,
        )
        state = torch.load(
            f"{save_dir}/{safe}_model.pt", map_location="cpu", weights_only=True)
        model.load_state_dict(state)
        model.eval()

        loaded[target] = {
            "model": model, "feat_mean": feat_mean, "feat_std": feat_std,
            "feats": feats, "preds": preds_df,
        }
        print(f"Loaded {target}: feats={len(feats)}, preds={len(preds_df)}")
    return loaded

## **Caution**: The following training loop can take a long time to run

In [ ]:
fold_results         = {}
best_params_by_target = {}
feat_cols_by_target   = {}

RUN_TS  = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = f"{SAVE_DIR}/{RUN_TS}"
os.makedirs(run_dir, exist_ok=True)

for target in tqdm(TARGET_TICKERS, desc="targets"):
    data = datasets[target]
    n    = len(data)
    initial_train = min(int(252 * INITIAL_TRAIN_YEARS), n // 2)
    folds = expanding_window_folds(n, initial_train, TEST_SIZE, EMBARGO)
    print(f"[{target}] n={n}, folds={len(folds)}")

    # ── Global feature selection (once on initial train period) ──────────────
    init_df   = data.iloc[:initial_train]
    X_init    = init_df.drop(columns=["target"])
    y_init    = init_df["target"]
    feat_cols, importances = select_features(X_init, y_init)
    feat_cols_by_target[target] = feat_cols

    importances.sort_values().plot.barh(
        figsize=(8, max(3, len(importances) * 0.35)),
        title=f"{target} — XGBoost Feature Importance (global selection)",
        color="steelblue",
    )
    plt.xlabel("Importance"); plt.tight_layout(); plt.show()
    print(f"[{target}] selected features ({len(feat_cols)}): {feat_cols}")

    # ── Optuna HPO ────────────────────────────────────────────────────────────
    study       = tune_with_optuna(data, folds, feat_cols, n_trials=OPTUNA_TRIALS)
    best_params = study.best_params
    best_params_by_target[target] = best_params
    print(f"[{target}] best params: {best_params}")

    # ── Walk-forward training ─────────────────────────────────────────────────
    (preds_df, last_model, last_mean, last_std,
     fold_hist, train_losses, val_losses) = walk_forward(
        data, best_params, folds, feat_cols, desc=f"{target} folds")

    mse = float(((preds_df["pred"] - preds_df["truth"]) ** 2).mean())
    hit = float(((preds_df["pred"] * preds_df["truth"]) > 0).mean())

    result = {
        "preds":        preds_df,
        "mse":          mse,
        "hit":          hit,
        "last_model":   last_model,
        "last_mean":    last_mean,
        "last_std":     last_std,
        "last_feats":   feat_cols,
        "fold_history": fold_hist,
        "train_losses": train_losses,
        "val_losses":   val_losses,
    }
    fold_results[target] = result
    save_target(target, result, best_params, save_dir=run_dir)

    # ── Training history (last fold) ─────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(train_losses, label="Train Loss")
    ax.plot(val_losses,   label="Val Loss")
    ax.set_title(f"{target} — Loss History (last fold)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Huber Loss")
    ax.legend(); ax.grid(True)
    plt.tight_layout()
    plt.savefig(f"{run_dir}/{target.replace('.','_')}_loss_history.png", dpi=100)
    plt.show()

    print(f"[{target}] walk-forward: n={len(preds_df)}, mse={mse:.6f}, hit={hit:.3f}\n")

summary_df = pd.DataFrame(
    {t: {"n": len(r["preds"]), "mse": r["mse"], "hit": r["hit"]}
     for t, r in fold_results.items()}
).T

with open(f"{run_dir}/best_params.json", "w") as f:
    json.dump(best_params_by_target, f, indent=2)
with open(f"{run_dir}/feat_cols.json", "w") as f:
    json.dump(feat_cols_by_target, f, indent=2)
summary_df.to_csv(f"{run_dir}/summary.csv")
print(f"Summary saved to {run_dir}/")
summary_df

In [ ]:
def predict_next(model, feat_mean, feat_std, feats, recent_data,
                 lookback=LOOKBACK, vol_window=VOL_WINDOW, device="cpu"):
    """v3: Predict volatility-normalized t+HORIZON log return and de-normalize.

    Parameters
    ----------
    recent_data : pd.DataFrame
        Raw (unscaled) recent data. Must contain either a "close" column or a
        column whose name contains "Price Close" so realised volatility can be
        computed for de-normalization.

    Returns
    -------
    pred_return : float
        Estimated cumulative log return over next HORIZON days.
    pred_sharpe : float
        Raw model output (volatility-normalized return).
    """
    if len(recent_data) < lookback:
        raise ValueError(f"Need at least {lookback} rows, got {len(recent_data)}")

    window = recent_data[feats].iloc[-lookback:].values.astype(np.float32)
    scaled = ((window - feat_mean) / feat_std).astype(np.float32)
    x = torch.tensor(scaled[None, :, :]).to(device)
    model.to(device).eval()
    with torch.no_grad():
        pred = model(x)
    pred_sharpe = float(pred.cpu().item())

    if "close" in recent_data.columns:
        close = recent_data["close"].astype(float)
    else:
        close_col = [c for c in recent_data.columns if "Price Close" in c][0]
        close = recent_data[close_col].astype(float)

    ret_1d  = np.log(close / close.shift(1))
    cur_vol = ret_1d.iloc[-vol_window:].std()
    pred_return = pred_sharpe * float(cur_vol)

    return pred_return, pred_sharpe


In [ ]:
# load_results() usage example (run in a new session after training):
#
# run_dir = "../model/20260419_143022"   # replace with your timestamped folder
# loaded  = load_results(run_dir)
# pred = predict_next(
#     loaded["SPY"]["model"], loaded["SPY"]["feat_mean"], loaded["SPY"]["feat_std"],
#     loaded["SPY"]["feats"], recent_data=datasets["SPY"].drop(columns=["target"])
# )

# Validation

In [ ]:
def validation_metrics(preds_df):
    p = preds_df["pred"].values
    t = preds_df["truth"].values
    return {
        "n":           len(p),
        "mse":         float(((p - t) ** 2).mean()),
        "mae":         float(np.abs(p - t).mean()),
        "rmse":        float(np.sqrt(((p - t) ** 2).mean())),
        "ic_spearman": float(pd.Series(p).corr(pd.Series(t), method="spearman")),
        "ic_pearson":  float(pd.Series(p).corr(pd.Series(t), method="pearson")),
        "hit_ratio":   float(((p * t) > 0).mean()),
    }

In [ ]:
validation_df = pd.DataFrame(
    {t: validation_metrics(r["preds"]) for t, r in fold_results.items()}
).T
print(validation_df.round(5))

# (1) IC
validation_df[["ic_spearman", "ic_pearson"]].plot.bar(
    figsize=(8, 4), title="Information Coefficient per target")
plt.axhline(0, color="k", lw=0.5); plt.tight_layout(); plt.show()

# (2) Hit ratio
validation_df[["hit_ratio"]].plot.bar(
    figsize=(8, 4), title="Hit Ratio per target")
plt.axhline(0.5, color="k", ls="--", lw=0.5); plt.tight_layout(); plt.show()

# (3) Predicted vs Actual scatter
fig, axes = plt.subplots(1, len(TARGET_TICKERS), figsize=(4 * len(TARGET_TICKERS), 4))
axes = axes if len(TARGET_TICKERS) > 1 else [axes]
for ax, t in zip(axes, TARGET_TICKERS):
    df_p = fold_results[t]["preds"]
    ax.scatter(df_p["truth"], df_p["pred"], s=4, alpha=0.4)
    lim = max(df_p["truth"].abs().max(), df_p["pred"].abs().max())
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.5)
    ax.set_title(t); ax.set_xlabel("Truth"); ax.set_ylabel("Pred")
plt.suptitle("Predicted vs Actual"); plt.tight_layout(); plt.show()

# (4) Fold-level Hit ratio stability
fig, ax = plt.subplots(figsize=(10, 4))
for t in TARGET_TICKERS:
    fh = fold_results[t]["fold_history"].set_index("test_start")
    fh["hit"].plot(ax=ax, marker="o", ms=3, label=t)
ax.axhline(0.5, color="k", ls="--", lw=0.5)
ax.set_title("Hit ratio across folds"); plt.legend(); plt.tight_layout(); plt.show()

validation_df.round(5)

In [ ]:
# Actual vs. Predicted time series per target
try:
    _results = fold_results
    _run_dir = run_dir
except NameError:
    _run_dir = "../model/YYYYMMDD_HHMMSS"   # ← replace with your folder
    _results = load_results(_run_dir)

for target, r in _results.items():
    preds_df = r["preds"]
    safe = target.replace(".", "_")
    ax = preds_df.plot(
        y=["truth", "pred"], figsize=(12, 3), alpha=0.7,
        title=f"{target} — Actual vs. Predicted (t+1 log return)")
    ax.set_xlabel("Date")
    ax.axhline(0, color="k", lw=0.5, ls="--")
    plt.tight_layout()
    plt.savefig(f"{_run_dir}/{safe}_pred_vs_actual.png", dpi=100)
    plt.show()

# Comprehensive Evaluation

In [ ]:
def comprehensive_evaluation(target, preds_df, save_dir=None):
    """
    4-panel evaluation: scatter, residual distribution, residuals over time, Q-Q plot.
    Adapted from the original Transformer notebook for log-return prediction.
    """
    p = preds_df["pred"].values
    t = preds_df["truth"].values
    residuals = t - p

    rmse = float(np.sqrt(((p - t) ** 2).mean()))
    mae  = float(np.abs(p - t).mean())
    ic   = float(pd.Series(p).corr(pd.Series(t), method="spearman"))
    hit  = float(((p * t) > 0).mean())

    print(f"\n{'='*55}")
    print(f"  {target} — Comprehensive Evaluation")
    print(f"{'='*55}")
    print(f"  RMSE:           {rmse:.6f}")
    print(f"  MAE:            {mae:.6f}")
    print(f"  IC (Spearman):  {ic:.4f}")
    print(f"  Hit Ratio:      {hit:.3f}")
    print(f"{'='*55}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # (1) Scatter: Actual vs Predicted
    lim = max(np.abs(t).max(), np.abs(p).max()) * 1.05
    axes[0, 0].scatter(t, p, s=4, alpha=0.4, color="steelblue")
    axes[0, 0].plot([-lim, lim], [-lim, lim], "r--", lw=1.5,
                    label="Perfect prediction")
    axes[0, 0].set_xlabel("Actual log return")
    axes[0, 0].set_ylabel("Predicted log return")
    axes[0, 0].set_title("Actual vs Predicted")
    axes[0, 0].legend(); axes[0, 0].grid(True)

    # (2) Residual distribution
    axes[0, 1].hist(residuals, bins=50, edgecolor="black",
                    color="steelblue", alpha=0.8)
    axes[0, 1].axvline(0, color="r", ls="--", lw=1.5)
    axes[0, 1].set_xlabel("Residual (Actual − Predicted)")
    axes[0, 1].set_ylabel("Frequency")
    axes[0, 1].set_title(
        f"Residual Distribution (mean={residuals.mean():.4f}, std={residuals.std():.4f})")
    axes[0, 1].grid(True)

    # (3) Residuals over time
    axes[1, 0].plot(preds_df.index, residuals, lw=0.8, alpha=0.8)
    axes[1, 0].axhline(0, color="r", ls="--", lw=1.5)
    axes[1, 0].set_xlabel("Date"); axes[1, 0].set_ylabel("Residual")
    axes[1, 0].set_title("Residuals Over Time"); axes[1, 0].grid(True)
    axes[1, 0].xaxis.set_major_locator(plt.MaxNLocator(10))
    axes[1, 0].tick_params(axis="x", rotation=45)

    # (4) Q-Q plot
    stats.probplot(residuals, dist="norm", plot=axes[1, 1])
    axes[1, 1].set_title("Q-Q Plot (Normality Check)"); axes[1, 1].grid(True)

    plt.suptitle(f"{target} — Comprehensive Evaluation", fontsize=13, y=1.01)
    plt.tight_layout()
    if save_dir:
        safe = target.replace(".", "_")
        plt.savefig(f"{save_dir}/{safe}_comprehensive_eval.png", dpi=100)
    plt.show()

    return {"rmse": rmse, "mae": mae, "ic_spearman": ic, "hit_ratio": hit}

In [ ]:
comp_results = {}
for target, r in _results.items():
    comp_results[target] = comprehensive_evaluation(
        target, r["preds"], save_dir=_run_dir)

comp_df = pd.DataFrame(comp_results).T
print("\nComprehensive Evaluation Summary:")
print(comp_df.round(5))

# Backtesting

In [ ]:
def backtest(preds_df, threshold=0.0):
    p        = preds_df["pred"]
    t        = preds_df["truth"]
    position = np.sign(p)
    if threshold > 0:
        position = position.where(p.abs() > threshold, 0)
    strat_ret = position * t
    bench_ret = t
    return pd.DataFrame({
        "position":  position,
        "strat_ret": strat_ret,
        "bench_ret": bench_ret,
        "strat_eq":  np.exp(strat_ret.cumsum()),
        "bench_eq":  np.exp(bench_ret.cumsum()),
    }, index=preds_df.index)

def perf_metrics(ret, periods=252):
    ret = ret.dropna()
    if len(ret) == 0:
        return {}
    equity  = np.exp(ret.cumsum())
    ann_ret = ret.mean() * periods
    ann_vol = ret.std() * np.sqrt(periods)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0.0
    downside = ret[ret < 0].std() * np.sqrt(periods)
    sortino  = ann_ret / downside if downside > 0 else 0.0
    dd       = float((equity / equity.cummax() - 1).min())
    total_ret = float(equity.iloc[-1]) - 1
    years    = len(ret) / periods
    cagr     = (1 + total_ret) ** (1 / years) - 1 if years > 0 else 0.0
    calmar   = cagr / abs(dd) if dd < 0 else 0.0
    return {
        "total_return": float(total_ret),
        "cagr":         float(cagr),
        "ann_vol":      float(ann_vol),
        "sharpe":       float(sharpe),
        "sortino":      float(sortino),
        "max_drawdown": dd,
        "calmar":       float(calmar),
        "win_rate":     float((ret > 0).mean()),
    }

In [ ]:
backtest_results = {t: backtest(r["preds"]) for t, r in _results.items()}

perf_rows = []
for t, bt in backtest_results.items():
    strat = perf_metrics(bt["strat_ret"])
    bench = perf_metrics(bt["bench_ret"])
    row = {"target": t}
    row.update({f"strat_{k}": v for k, v in strat.items()})
    row.update({f"bench_{k}": v for k, v in bench.items()})
    perf_rows.append(row)
perf_df = pd.DataFrame(perf_rows).set_index("target")
print(perf_df.round(4))

# Equity curves + drawdowns per target
fig, axes = plt.subplots(len(TARGET_TICKERS), 2,
                         figsize=(14, 3 * len(TARGET_TICKERS)))
axes = axes.reshape(len(TARGET_TICKERS), 2)
for i, t in enumerate(TARGET_TICKERS):
    bt = backtest_results[t]
    bt[["strat_eq", "bench_eq"]].rename(
        columns={"strat_eq": "Strategy", "bench_eq": "Buy & Hold"}
    ).plot(ax=axes[i, 0], logy=True, title=f"{t} — Equity (log scale)")
    pd.DataFrame({
        "Strategy":    bt["strat_eq"]  / bt["strat_eq"].cummax()  - 1,
        "Buy & Hold":  bt["bench_eq"]  / bt["bench_eq"].cummax()  - 1,
    }).plot.area(ax=axes[i, 1], alpha=0.4, title=f"{t} — Drawdown")
plt.tight_layout(); plt.show()

# Sharpe comparison
perf_df[["strat_sharpe", "bench_sharpe"]].rename(
    columns={"strat_sharpe": "Strategy", "bench_sharpe": "Buy & Hold"}
).plot.bar(figsize=(9, 4), title="Sharpe: Strategy vs Buy & Hold")
plt.tight_layout(); plt.show()

# Max Drawdown comparison
perf_df[["strat_max_drawdown", "bench_max_drawdown"]].rename(
    columns={"strat_max_drawdown": "Strategy", "bench_max_drawdown": "Buy & Hold"}
).plot.bar(figsize=(9, 4), title="Max Drawdown: Strategy vs Buy & Hold")
plt.tight_layout(); plt.show()

perf_df.round(4)